# Stage1 成分描述量来源预检

按冻结 `source_preflight_plan.json` 检查 Stage1 原训练/评价日历与NAS归档的成分描述量覆盖。只读 Qlib Recorder，不读收益、拟合或回测。

In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
import qlib
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_regime_feature_direction_20261022'
plan=json.loads((card/'source_preflight_plan.json').read_text())
summaries={}
for key,src in plan['sources'].items():
    s=json.loads((root/src['card']/'summary.json').read_text())
    assert s['recorder_id']==src['recorder_id']
    manifest=json.loads((root/src['card']/'verification_manifest.json').read_text())
    entries=[e for e in manifest['entries'] if e['path'].endswith('/artifacts/'+src['artifact'])]
    assert len(entries)==1 and entries[0]['sha256']==src['sha256']
    f=root/src['card']/entries[0]['path']
    assert hashlib.sha256(f.read_bytes()).hexdigest()==src['sha256']
    summaries[key]=s
qlib.init(provider_uri=summaries['training_calendar']['provider_uri'],region='us')
objects={}
for key,src in plan['sources'].items():
    s=summaries[key]
    with R.uri_context(s['recorder_uri']):
        rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
        objects[key]=rec.load_object(src['artifact'])
print('Three source identities and Qlib Recorder objects verified.')

In [2]:
train=objects['training_calendar']
eval_frame=objects['stage1_evaluation']
old,fresh=objects['descriptors']['old'],objects['descriptors']['fresh']
assert old.index.is_monotonic_increasing and fresh.index.is_monotonic_increasing
cut=pd.Timestamp('2026-07-02')
assert old.index.max()==cut
assert fresh.index.min()<=cut and fresh.index.max()==pd.Timestamp('2026-09-30')
combined=pd.concat([old.loc[old.index<=cut],fresh.loc[fresh.index>cut]])
assert combined.index.is_unique and combined.index.is_monotonic_increasing
train_dates=train.index[train.index<pd.Timestamp('2023-01-01')]
eval_dates=pd.DatetimeIndex(eval_frame.signal_date)
assert len(eval_dates)==944 and eval_dates.is_unique
fields=['new_low_level20','new_low_price_contraction5','downside_top5_share']
counts={}
for label,dates in [('train',train_dates),('evaluation',eval_dates)]:
    joined=combined.reindex(dates)[fields]
    missing=dates[joined.new_low_level20.isna()]
    counts[label]={'calendar_rows':len(dates),'first_date':str(dates.min().date()),'last_date':str(dates.max().date()),
                   'known':{name:int(joined[name].notna().sum()) for name in fields},
                   'all_three_known':int(joined.notna().all(axis=1).sum()),
                   'new_low_missing_count':len(missing),'new_low_missing_first5':[str(d.date()) for d in missing[:5]],
                   'new_low_missing_last5':[str(d.date()) for d in missing[-5:]]}
    if label=='evaluation':counts[label]['new_low_missing_dates']=[str(d.date()) for d in missing]
assert counts['train']['calendar_rows']==3020 and counts['train']['all_three_known']==122
assert counts['evaluation']['calendar_rows']==944 and counts['evaluation']['all_three_known']==871
assert counts['evaluation']['new_low_missing_dates']==['2026-10-01','2026-10-02','2026-10-05','2026-10-06','2026-10-07']
result={'status':'needs_source_history','execution_authorized':False,'source_plan_sha256':hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest(),
        'source_recorder_ids':{k:v['recorder_id'] for k,v in summaries.items()},'cutover_after':'2026-07-02',
        'coverage':counts,'historical_first_receipt_certified':False,'model_fits':0,'native_accounts':0,'outcome_reads':0,
        'decision':'Stage1 full training has only 122/3020 complete descriptor rows; evaluation has 871/944, with five signal dates after the latest source. Do not fit on a truncated or backfilled population.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2))

## 中文结论

来源身份与覆盖已核对。Stage1训练日历3020日中三项描述量同时已知仅122日（均集中在2022年后段）；评价944日中871日同时已知，2026年10月1日至7日尚无新低水平值。历史首次接收时点也未认证。状态为 `needs_source_history`，本候选不进入拟合或完整账户；不能以缺失值伪造旧成分历史，也不能因为历史回看在坏季度有覆盖就称有预测信息。下一步寻找覆盖Stage1训练期的独立信息，或取得原始更早的披露和价格来源后重新资格核查。